<a href="https://colab.research.google.com/github/TheFaridKarami/farid-ai-lab/blob/main/projects/nlp/wnut17-robust-ner/notebooks/wnut17_multi_model_ner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
! pip install -q seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
import sys
import numpy as np
import torch, time
from torch import nn
import transformers
from transformers import (AutoTokenizer,
                          AutoModelForTokenClassification,
                          DataCollatorForTokenClassification,
                          TrainingArguments,
                          Trainer,
                          set_seed)

import datasets
from seqeval.metrics import classification_report, accuracy_score
from seqeval.scheme import IOB2

In [4]:
wnut = datasets.load_dataset("flaitenberger/wnut_17")

README.md:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  307kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 66.4kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  115kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3394 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1009 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1287 [00:00<?, ? examples/s]

In [5]:
label_list = [label for label in wnut["train"].features["ner_tags"].feature.names]
id2label =   {i:label for i, label in enumerate(label_list)}
label2id =   {label:i for i, label in id2label.items()}


In [6]:
shown = 0
for ex in wnut["train"]:

  if any(t != 0 for t in ex["ner_tags"]):
    print(" ".join(ex["tokens"]))
    print([(tok, id2label[t]) for tok, t in zip(ex["tokens"], ex["ner_tags"])])
    print()
  shown += 1

  if shown == 5:
    break

@paulwalk It 's the view from where I 'm living for two weeks . Empire State Building = ESB . Pretty bad storm here last evening .
[('@paulwalk', 'O'), ('It', 'O'), ("'s", 'O'), ('the', 'O'), ('view', 'O'), ('from', 'O'), ('where', 'O'), ('I', 'O'), ("'m", 'O'), ('living', 'O'), ('for', 'O'), ('two', 'O'), ('weeks', 'O'), ('.', 'O'), ('Empire', 'B-location'), ('State', 'I-location'), ('Building', 'I-location'), ('=', 'O'), ('ESB', 'B-location'), ('.', 'O'), ('Pretty', 'O'), ('bad', 'O'), ('storm', 'O'), ('here', 'O'), ('last', 'O'), ('evening', 'O'), ('.', 'O')]

From Green Newsfeed : AHFA extends deadline for Sage Award to Nov . 5 http://tinyurl.com/24agj38
[('From', 'O'), ('Green', 'O'), ('Newsfeed', 'O'), (':', 'O'), ('AHFA', 'B-group'), ('extends', 'O'), ('deadline', 'O'), ('for', 'O'), ('Sage', 'O'), ('Award', 'O'), ('to', 'O'), ('Nov', 'O'), ('.', 'O'), ('5', 'O'), ('http://tinyurl.com/24agj38', 'O')]

Pxleyes Top 50 Photography Contest Pictures of August 2010 ... http://bit.ly/b

In [7]:
for split in wnut:
  lens = [len(ex["tokens"]) for ex in wnut[split]]
  print(split, "mean:",round(np.mean(lens),2), "max: ", np.max(lens), "median: ", np.median(lens), "95th pct: ", np.quantile(lens, 0.95))

train mean: 18.48 max:  41 median:  18.0 95th pct:  30.0
validation mean: 15.59 max:  82 median:  13.0 95th pct:  38.0
test mean: 18.18 max:  105 median:  15.0 95th pct:  42.0


In [8]:
from collections import Counter

for split in wnut:
  c = Counter(id2label[t] for ex in wnut[split] for t in ex["ner_tags"])
  ents = {l[2:]: n for l , n in c.items() if l.startswith("B-")}
  total = sum(c.values())
  print(split, "tokens:", total, "O share:", round(c["O"] / total, 3))
  print(ents, "total entities:", sum(ents.values()))

train tokens: 62730 O share: 0.95
{'location': 548, 'group': 264, 'corporation': 221, 'person': 660, 'creative-work': 140, 'product': 142} total entities: 1975
validation tokens: 15733 O share: 0.921
{'location': 74, 'creative-work': 105, 'person': 470, 'product': 114, 'group': 39, 'corporation': 34} total entities: 836
test tokens: 23394 O share: 0.926
{'location': 150, 'group': 165, 'person': 429, 'creative-work': 142, 'corporation': 66, 'product': 127} total entities: 1079


In [9]:
tok = AutoTokenizer.from_pretrained("bert-base-cased")

words = ["Pxleyes", "Top", "50", "Photography", "Contest", "#photography"]

enc = tok(words, is_split_into_words= True)

print(tok.convert_ids_to_tokens(enc["input_ids"]))
print(enc.word_ids())

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

['[CLS]', 'P', '##xley', '##es', 'Top', '50', 'Photography', 'Contest', '#', 'photography', '[SEP]']
[None, 0, 0, 0, 1, 2, 3, 4, 5, 5, None]


In [10]:
print(enc.word_ids())
print(wnut["train"]["ner_tags"])

[None, 0, 0, 0, 1, 2, 3, 4, 5, 5, None]
Column([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 8, 8, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0], [9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ...])


In [11]:
def tokenize_and_align(batch):

  enc = tok(batch["tokens"], is_split_into_words=True, truncation=True, max_length=256)
  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):
    prev = None
    labs = []
    for w in enc.word_ids(batch_index=i):
      if w is None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)
  enc["labels"] = all_labels

  return enc

In [12]:
tokenized = wnut.map(tokenize_and_align, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized[split]["labels"]))
  print(split, "max_tokens:", max(len(x) for x in tokenized[split]["input_ids"]), "words lost:", lost)

Map:   0%|          | 0/3394 [00:00<?, ? examples/s]

Map:   0%|          | 0/1009 [00:00<?, ? examples/s]

Map:   0%|          | 0/1287 [00:00<?, ? examples/s]

train max_tokens: 90 words lost: 0
validation max_tokens: 84 words lost: 0
test max_tokens: 198 words lost: 0


In [13]:
val_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [14]:
model = AutoModelForTokenClassification.from_pretrained("bert-base-cased", num_labels=len(label_list), id2label=id2label, label2id=label2id)
print(model.classifier)
print(sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad))

collator = DataCollatorForTokenClassification(tok)
batch = collator([tokenized["train"][i] for i in range(3)])
print({k: v.shape for k, v in batch.items()})
print(batch["labels"][0])

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Linear(in_features=768, out_features=13, bias=True)
107729677 107729677
{'input_ids': torch.Size([3, 35]), 'token_type_ids': torch.Size([3, 35]), 'attention_mask': torch.Size([3, 35]), 'labels': torch.Size([3, 35])}
tensor([-100,    0, -100, -100, -100,    0,    0, -100,    0,    0,    0,    0,
           0,    0, -100,    0,    0,    0,    0,    0,    7,    8,    8,    0,
           7, -100,    0,    0,    0,    0,    0,    0,    0,    0, -100])


In [15]:
def evaluate_ner(gold, pred):

    rep = classification_report(
        gold,
        pred,
        mode="strict",
        scheme=IOB2,
        output_dict=True,
        zero_division=0)

    out = {"precision": rep["micro avg"]["precision"],
           "recall": rep["micro avg"]["recall"],
           "f1": rep["micro avg"]["f1-score"],
           "token_acc": accuracy_score(gold, pred)}

    out["per_type_f1"] = {k: v["f1-score"] for k, v in rep.items() if "avg" not in k}

    return out

In [16]:
def to_tags(logits, labels):

  preds = np.argmax(logits, axis = -1)
  gold, pred = [], []
  for pr, lb in zip(preds, labels):
    gold.append([id2label[l] for l in lb if l != -100])
    pred.append([id2label[q] for q, l in zip(pr, lb) if l != -100])

  return gold, pred

def compute_metrics(p):
  gold, pred = to_tags(*p)
  r = evaluate_ner(gold, pred)
  return {
      "precision":r["precision"],
      "f1":r["f1"],
      "recall": r["recall"],
      "accuracy": r["token_acc"]}




In [17]:
b = collator([tokenized["validation"][i] for i in range(64)])
labels = b["labels"].numpy()
logits = np.eye(len(label_list))[np.where(labels == -100, 0, labels)]
print(compute_metrics((logits, labels)))

{'precision': np.float64(1.0), 'f1': np.float64(1.0), 'recall': np.float64(1.0), 'accuracy': 1.0}


In [20]:
import torch, time
from transformers import Trainer, TrainingArguments, set_seed

set_seed(42)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-cased",
    num_labels = len(label_list),
    id2label = id2label,
    label2id = label2id)

args = TrainingArguments(
    output_dir="bert_base_cased_ckpt",
    learning_rate = 5e-5,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    num_train_epochs = 10,
    weight_decay = 0.01,
    warmup_steps = 107,
    fp16  =True,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    save_total_limit = 1,
    load_best_model_at_end = True,
    metric_for_best_model = "f1",
    seed = 42,
    report_to = "none")


trainer = Trainer(model = model,
                  args = args,
                  train_dataset = tokenized["train"],
                  eval_dataset  = tokenized["validation"],
                  data_collator = collator,
                  processing_class = tok,
                  compute_metrics = compute_metrics)

# torch.cuda.reset_peak_memory_stats()
# t0 = time.time()

# trainer.train()


# train_time = time.time() - t0
# peak_gb = torch.cuda.max_memory_allocated() / 1e9
# print("train time (s):", round(train_time), "peak GPU memory (GB):", round(peak_gb, 2))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

In [21]:
out = trainer.predict(tokenized["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer.state.best_metric, trainer.state.best_model_checkpoint)

{'precision': np.float64(0.004631058968817536), 'recall': np.float64(0.017942583732057416), 'f1': np.float64(0.007361963190184048), 'token_acc': 0.022754719379647875, 'per_type_f1': {'corporation': np.float64(0.0), 'creative-work': np.float64(0.0), 'group': np.float64(0.0), 'location': np.float64(0.02072538860103627), 'person': np.float64(0.0034542314335060447), 'product': np.float64(0.011428571428571429)}}
True
None None


#**Architecture: RoBERTa-base**

In [43]:
MODEL_NAME = "FacebookAI/roberta-base"
tok_rb = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space = True)
enc = tok_rb(words, is_split_into_words= True)

In [44]:
def tokenize_and_align_rb(batch):
  enc = tok_rb(batch["tokens"],
               is_split_into_words= True,
               truncation= True,
               max_length = 256)

  all_labels = []
  for i, tags in enumerate(batch["ner_tags"]):

    prev = None
    labs = []
    for w in enc.word_ids(batch_index = i):

      if w == None:
        labs.append(-100)
      elif w != prev:
        labs.append(tags[w])
      else:
        labs.append(-100)
      prev = w

    all_labels.append(labs)

  enc["labels"] = all_labels

  return enc


In [55]:
tokenized_rb = wnut.map(tokenize_and_align_rb, batched = True, remove_columns = wnut["train"].column_names)

for split in tokenized_rb:
  lost = sum(len(ex["tokens"]) - sum(l != -100 for l in lab) for ex, lab in zip(wnut[split], tokenized_rb[split]["labels"]))
  print(split, "max tokens", max(len(x) for x in tokenized_rb[split]["input_ids"]), "words lost", lost)

train max tokens 70 words lost 0
validation max tokens 84 words lost 0
test max tokens 175 words lost 0


In [46]:
val_gold =  [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["validation"]]
test_gold = [[id2label[t] for t in ex["ner_tags"]] for ex in wnut["test"]]

In [47]:
MODEL_NAME = "FacebookAI/roberta-base"

model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME,
                                                        num_labels = len(label_list),
                                                        id2label= id2label,
                                                        label2id = label2id)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: FacebookAI/roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
classifier.weight         | MISSING    | 
classifier.bias           | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [48]:
print(model.classifier)
print()
print("parameters without gradient",sum(p.numel() for p in model.parameters()))
print()
print("Parameters with gradient",sum(p.numel() for p in model.parameters() if p.requires_grad))

Linear(in_features=768, out_features=13, bias=True)

parameters without gradient 124065037

Parameters with gradient 124065037


In [49]:
collator = DataCollatorForTokenClassification(tok_rb)

In [54]:
import torch, time
from transformers import Trainer, TrainingArguments, set_seed, EarlyStoppingCallback

args = TrainingArguments(
    output_dir = "roberta_base_ckpt",
    num_train_epochs = 20,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size = 64,
    learning_rate = 5e-5,
    warmup_steps = 107,
    weight_decay = 0.01,
    save_strategy= "epoch",
    eval_strategy = "epoch",
    seed= 42,
    metric_for_best_model = "f1",
    report_to = "none",
    load_best_model_at_end = True,
    save_total_limit = 1,
    fp16 = True,
    greater_is_better = True
)

trainer = Trainer(
    model = model,
    args = args,
    data_collator = collator,
    train_dataset= tokenized_rb["train"],
    eval_dataset= tokenized_rb["validation"],
    processing_class= tok_rb,
    compute_metrics = compute_metrics,
    callbacks = [
        EarlyStoppingCallback(
        early_stopping_patience = 2,
        early_stopping_threshold = 0.001
        )
    ])

torch.cuda.reset_peak_memory_stats()

t0 = time.time()

trainer.train()

train_time = time.time() - t0

peak_gb = torch.cuda.max_memory_allocated() / 1e9

print("train time (s)", train_time, "peak GPU memory", peak_gb)

Epoch,Training Loss,Validation Loss,Precision,F1,Recall,Accuracy
1,No log,0.216257,0.703154,0.551273,0.453349,0.947245
2,No log,0.183340,0.763025,0.634521,0.543062,0.959639
3,No log,0.213291,0.776340,0.610022,0.502392,0.957859
4,No log,0.206403,0.712166,0.635762,0.574163,0.960020
5,0.181802,0.240903,0.733652,0.628845,0.550239,0.959830
6,0.181802,0.280013,0.763699,0.628169,0.533493,0.959003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

train time (s) 132.51495552062988 peak GPU memory 2.655504384


In [58]:
out = trainer.predict(tokenized_rb["validation"])
gold, pred = to_tags(out.predictions, out.label_ids)
print(evaluate_ner(gold, pred))
print(gold == val_gold)
print(trainer.state.best_metric, trainer.state.best_model_checkpoint)

{'precision': np.float64(0.712166172106825), 'recall': np.float64(0.5741626794258373), 'f1': np.float64(0.6357615894039735), 'token_acc': 0.9600203394139706, 'per_type_f1': {'corporation': np.float64(0.4666666666666667), 'creative-work': np.float64(0.3195266272189349), 'group': np.float64(0.2413793103448276), 'location': np.float64(0.6993006993006994), 'person': np.float64(0.7731843575418994), 'product': np.float64(0.38918918918918916)}}
True
0.6357615894039735 roberta_base_ckpt/checkpoint-428


# **Distilbert Base Cased**

In [60]:
MODEL_NAME = "distilbert-base-cased"
tok_dis = AutoTokenizer.from_pretrained(MODEL_NAME)


config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]